**BfPortf 03 Metrics and strategies**

This example goes through the parts that `01` and `02` only touch:

- the metric registry and the optimization direction (`maximize`),
- `tolerance`: how much improvement is required to add the next symbol,
- the cost of `GreedySearch`, `BeamSearch` and `ExhaustiveSearch`,
- the errors raised for an unknown metric, an unknown symbol or an exhausted budget,
- short legs with `add_inverted_return`.

The examples need the `examples` extra (`uv sync --extra examples`).

In [ ]:
# Import section
import time
import urllib.request
from pathlib import Path

import pandas as pd

import bfportf as bf

pd.set_option('display.precision', 3)

### Load data

`prices.parquet` is not tracked in git: the local copy from `assets/` is used when it exists, otherwise
the file is downloaded from the release asset.

In [ ]:
# Load data
# prices.parquet (4.5 MB, 500 symbols) is published as a GitHub release asset and is not tracked in git.
# A local copy is preferred: the repository layout (assets/prices.parquet), a file next to the notebook,
# or ../assets when the kernel runs inside the examples directory.
PRICES_CANDIDATES = (
    Path('assets/prices.parquet'),
    Path('prices.parquet'),
    Path('../assets/prices.parquet'),
)
PRICES_URL = 'https://github.com/answering007/bfportf/releases/latest/download/prices.parquet'

local_path = next((path for path in PRICES_CANDIDATES if path.exists()), None)

if local_path is None:
    # No local copy: take the release asset and cache it next to the notebook
    local_path = Path('assets/prices.parquet') if Path('assets').is_dir() else Path('prices.parquet')
    try:
        with urllib.request.urlopen(PRICES_URL, timeout=120) as response:
            local_path.write_bytes(response.read())
    except Exception as error:
        raise RuntimeError(
            f'cannot download {PRICES_URL}, get the file manually: curl -L -o {local_path} {PRICES_URL}'
        ) from error
    source = f'{PRICES_URL} -> cached to {local_path}'
else:
    source = local_path

prices = pd.read_parquet(local_path)
if 'Date' in prices.columns:  # the file was written without an index
    prices = prices.set_index(pd.to_datetime(prices['Date'])).drop(columns=['Date'])

returns = bf.load_returns(prices)
bf.validate_returns(returns)
print(f'{returns.shape[0]} periods x {returns.shape[1]} symbols <- {source}')
returns.head()

### The metric registry

Every name of `bf.METRICS` is a `Metric`: the function, the direction and a one line description. The
`vectorized` flag tells whether the fast path (a whole candidate matrix at once) is available.

In [ ]:
registry = pd.DataFrame(
    {
        'maximize': {name: metric.maximize for name, metric in bf.METRICS.items()},
        'vectorized': {name: metric.vectorized is not None for name, metric in bf.METRICS.items()},
        'description': {name: metric.description for name, metric in bf.METRICS.items()},
    }
)
registry

### The direction comes from the metric

`max_drawdown` is a negative percentage with `maximize=True`, so "the largest value" is the smallest
drawdown and no extra flag is needed. The direction of a metric from the registry is used unless
`find_portfolios(maximize=...)` overrides it; a plain callable defaults to maximization.

In [ ]:
# Scale the whole history, then take small universes to keep the cells fast
scaled, weights = bf.scale_return_data(returns)
small = scaled[['SPY', 'AAPL', 'MSFT', 'GLD', 'TLT', 'XOM', 'JPM', 'KO', 'AMD', 'BA']]
medium = scaled.iloc[:, :50]

rows = []
for name in ('sharpe', 'sortino', 'max_drawdown', 'recovery_factor'):
    found = bf.find_portfolios(small, metric=name, custom_weights=weights, max_number=3, verbose=False)
    rows.append({'metric': name, 'value': found[0].value, 'portfolio': found[0].portfolio})

pd.DataFrame(rows).round({'value': 3})

### Flipping the direction

`resolve_metric` returns a copy of a registered metric with the direction overridden, which is handy to
look at the other end of the ranking.

In [ ]:
worst = bf.find_portfolios(
    small,
    metric=bf.resolve_metric('sharpe', maximize=False),
    custom_weights=weights,
    max_number=3,
    verbose=False,
)

print(worst[0].metric_name, round(worst[0].value, 3), worst[0].portfolio)

### Tolerance

`tolerance` is the minimal metric improvement required to accept the next symbol: positive values stop the
search earlier and produce shorter portfolios. The improvements on this 50 symbol universe are small (a few
hundredths of a Sharpe point per symbol), so a tolerance of 0.05 already changes the result.

In [ ]:
rows = []
for tolerance in (0.0, 0.05, 0.2):
    found = bf.find_portfolios(
        medium,
        metric='sharpe',
        custom_weights=weights,
        max_number=5,
        tolerance=tolerance,
        verbose=False,
    )
    rows.append(
        {
            'tolerance': tolerance,
            'symbols': len(found[0].symbols),
            'value': found[0].value,
            'portfolio': found[0].portfolio,
        }
    )

pd.DataFrame(rows).round({'value': 3})

### Strategies and their cost

`ExhaustiveSearch` is the reference implementation: on 50 symbols with `max_number=3` it evaluates every
combination and returns the true optimum. `BeamSearch(width=5)` looks at five partial portfolios per step
instead of one, `GreedySearch` is the default. The optimum may be the same, the cost is not.

In [ ]:
strategies = {
    'greedy': bf.GreedySearch(),
    'beam': bf.BeamSearch(width=5),
    'exhaustive': bf.ExhaustiveSearch(),
}

rows = []
for name, strategy in strategies.items():
    started = time.perf_counter()
    found = bf.find_portfolios(
        medium,
        metric='sharpe',
        custom_weights=weights,
        max_number=3,
        strategy=strategy,
        verbose=False,
    )
    rows.append(
        {
            'strategy': name,
            'value': found[0].value,
            'seconds': time.perf_counter() - started,
            'portfolio': found[0].portfolio,
        }
    )

pd.DataFrame(rows).round({'value': 3, 'seconds': 3})

### Errors

All package errors derive from `bf.BfPortfError`, so one `except` clause is enough. `ExhaustiveSearch`
refuses to run when the number of combinations exceeds `max_combinations`, which protects from an
accidental launch on a large universe (500 symbols with `max_number=3` is about 20 million combinations).

In [ ]:
def explain(call):
    """Run the call and print the error type and message."""
    try:
        call()
    except bf.BfPortfError as error:
        print(f'{type(error).__name__}: {error}')


# Unknown metric name: the message lists the available names
explain(lambda: bf.find_portfolios(small, metric='sharpe_ratio', max_number=3, verbose=False))

# Unknown target symbol
explain(lambda: bf.find_portfolios(small, metric='sharpe', target_symbols=['NOPE'], verbose=False))

# Exhaustive search budget
explain(
    lambda: bf.find_portfolios(
        scaled,
        metric='sharpe',
        max_number=3,
        strategy=bf.ExhaustiveSearch(max_combinations=100),
        verbose=False,
    )
)

### Short legs

`add_inverted_return` appends a negated copy of a column (`SPY-1`), so the search can pick a short leg as a
part of the portfolio. The inverted column is not covered by the inverse volatility weights, so this cell
runs the search without `custom_weights`: passing the original weights raises `UnknownSymbolError` for
`SPY-1`. Extend the weights with the inverted symbol (`weights['SPY-1'] = weights['SPY']`) when you need them.

In [ ]:
universe = small[['SPY', 'AAPL', 'MSFT', 'GLD', 'TLT']]
universe, inverted_names = bf.add_inverted_return(universe, ['SPY'])
print(f'inverted columns: {inverted_names}')

found = bf.find_portfolios(universe, metric='sharpe', max_number=3, verbose=False)
print(found[0].portfolio)
print(found[0].weights.round(3).to_dict())

`PortfolioResult.weights` always sums up to one (that is `normalized_weights`). Use
`get_portfolio_return(normalize="none")` when the original scale (the leverage implied by the weights) has
to be kept.

Example `02` applies the same search to a custom metric and compares the in-sample and the out-of-sample
report.